# 03 — RuASD CNN с локальным кэшем Colab
Версия notebook 02 с диагностикой и явным локальным кэшем `/content/ruasd_prepared_cache`. Исходные FLAC и checkpoints остаются на Google Drive. Копирование, обучение и `test_unseen` никогда не запускаются автоматически.

In [ ]:
# 1. Среда, репозиторий и зависимости
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/Unvand/audio-deepfake-detector.git'
REPO_DIR = Path('/content/audio-deepfake-detector')
if IN_COLAB:
    if not (REPO_DIR / '.git').exists():
        subprocess.run(['git', 'clone', '-q', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements-colab-train.txt')], check=True)
    os.chdir(REPO_DIR)
print('Python:', sys.version)

In [ ]:
# 2. Drive, GPU и исходный manifest (только чтение)
import pandas as pd
import torch
from IPython.display import clear_output, display

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
DRIVE_PREPARED_ROOT = Path('/content/drive/MyDrive/VKR_RuASD/prepared_v1')
LOCAL_CACHE_ROOT = Path('/content/ruasd_prepared_cache')
TRAINING_ROOT = Path('/content/drive/MyDrive/VKR_RuASD/training_v1')
RUNS_ROOT = TRAINING_ROOT / 'runs'
SEGMENTS_MANIFEST = DRIVE_PREPARED_ROOT / 'ruasd_segments_manifest.csv'
if not SEGMENTS_MANIFEST.exists():
    raise FileNotFoundError('Не найден подготовленный manifest: ' + str(SEGMENTS_MANIFEST))
if not torch.cuda.is_available():
    raise RuntimeError('Выберите Runtime → Change runtime type → GPU')
print('GPU:', torch.cuda.get_device_name(0))
segments = pd.read_csv(SEGMENTS_MANIFEST)
if segments['segment_id'].duplicated().any():
    raise ValueError('Найдены повторяющиеся segment_id')
if set(segments['split'].unique()) != {'train', 'validation', 'test_unseen'}:
    raise ValueError('Неожиданный набор split')
display(pd.crosstab(segments['split'], segments['label'], margins=True))
print('Сегментов:', len(segments))

## Загрузка данных для обучения
По умолчанию выбраны `train` и `validation`. Для копирования установите `RUN_CACHE_PREPARATION = True` в отдельной ячейке и запустите её через ▶. Выберите `test_unseen` отдельно перед финальной оценкой.

In [ ]:
# 3. Панель управления локальным кэшем
from dataclasses import asdict

import ipywidgets as widgets

if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()

from audio_deepfake_detector.src.colab_cache import (
    cache_space_report,
    check_local_cache,
    prepare_local_cache,
    require_ready_cache,
)
from audio_deepfake_detector.src.colab_tar_cache import (
    GIB,
    benchmark_tar_transfer,
    build_tar_archives,
    restore_tar_cache,
)

ARCHIVE_ROOT = DRIVE_PREPARED_ROOT.parent / 'prepared_v1_tar'
TAR_STAGING_ROOT = Path('/content/ruasd_tar_work')
cache_mode = widgets.Dropdown(options=[('Отдельные FLAC', 'files'), ('Готовые TAR', 'tar')], description='Способ')
drive_free_gib = widgets.FloatText(value=0.0, description='Квота Drive, ГиБ')
use_cache_widget = widgets.Checkbox(value=True, description='Использовать локальный кэш Colab')
train_split_widget = widgets.Checkbox(value=True, description='train')
validation_split_widget = widgets.Checkbox(value=True, description='validation')
test_split_widget = widgets.Checkbox(value=False, description='test_unseen')
checksum_widget = widgets.Checkbox(value=False, description='SHA-256 (медленнее)')
space_button = widgets.Button(description='Проверить свободное место')
check_button = widgets.Button(description='Проверить локальный кэш', button_style='info')
copy_progress = widgets.IntProgress(value=0, min=0, max=1, description='Файлы')
copy_stats = widgets.HTML(value='copied=0, skipped=0, errors=0')
cache_output = widgets.Output()
CACHE_VERIFIED_SPLITS = set()

def selected_cache_splits():
    selected = []
    if train_split_widget.value:
        selected.append('train')
    if validation_split_widget.value:
        selected.append('validation')
    if test_split_widget.value:
        selected.append('test_unseen')
    if not selected:
        raise ValueError('Выберите хотя бы один split')
    return tuple(selected)

def format_bytes(value):
    amount = float(value)
    for unit in ['Б', 'КиБ', 'МиБ', 'ГиБ', 'ТиБ']:
        if amount < 1024 or unit == 'ТиБ':
            return f'{amount:.2f} {unit}'
        amount /= 1024

def show_space(_):
    with cache_output:
        clear_output()
        try:
            report = cache_space_report(DRIVE_PREPARED_ROOT, LOCAL_CACHE_ROOT, selected_cache_splits())
            display(pd.Series({key: format_bytes(value) if key.endswith('bytes') else value for key, value in report.items()}))
            if not report['enough_space']:
                print('ПРЕДУПРЕЖДЕНИЕ: места недостаточно; копирование не запускайте.')
        except Exception as error:  # noqa: BLE001 — сообщение выводится в панели
            print('ОШИБКА ПРОВЕРКИ МЕСТА:', error)

def check_cache(_):
    with cache_output:
        clear_output()
        try:
            selected = selected_cache_splits()
            CACHE_VERIFIED_SPLITS.difference_update(selected)
            status = check_local_cache(
                DRIVE_PREPARED_ROOT, LOCAL_CACHE_ROOT, selected,
                verify_decode=True, verify_checksums=checksum_widget.value or cache_mode.value == 'tar',
                compare_source_sizes=cache_mode.value != 'tar',
            )
            display(pd.Series(asdict(status)))
            if status.ready:
                CACHE_VERIFIED_SPLITS.update(selected)
                print('Данные готовы для обучения с локального диска.')
        except Exception as error:  # noqa: BLE001 — сообщение выводится в панели
            print('ОШИБКА ПРОВЕРКИ КЭША:', error)

def update_copy_progress(current, total, stats):
    copy_progress.max = total
    copy_progress.value = current
    copy_stats.value = (f"copied={stats['copied']}, "
                        f"skipped={stats['skipped']}, errors={stats['errors']}")

def active_data_root(required_splits=('train', 'validation')):
    if not use_cache_widget.value:
        print('ПРЕДУПРЕЖДЕНИЕ: чтение выполняется непосредственно с Google Drive.')
        return DRIVE_PREPARED_ROOT
    required = set(required_splits)
    if not required.issubset(CACHE_VERIFIED_SPLITS):
        root = require_ready_cache(
            DRIVE_PREPARED_ROOT, LOCAL_CACHE_ROOT, required_splits,
            compare_source_sizes=cache_mode.value != 'tar',
        )
        CACHE_VERIFIED_SPLITS.update(required)
        return root
    quick_status = check_local_cache(
        DRIVE_PREPARED_ROOT, LOCAL_CACHE_ROOT, required_splits, verify_decode=False,
        compare_source_sizes=cache_mode.value != 'tar',
    )
    if not quick_status.ready:
        raise RuntimeError(quick_status.message)
    return LOCAL_CACHE_ROOT

space_button.on_click(show_space)
check_button.on_click(check_cache)
display(widgets.VBox([
    use_cache_widget,
    cache_mode, drive_free_gib,
    widgets.HTML('<b>Split для копирования</b>'),
    widgets.HBox([train_split_widget, validation_split_widget, test_split_widget]),
    checksum_widget, widgets.HBox([space_button, check_button]),
    copy_progress, copy_stats, cache_output,
]))

## Создание TAR на Drive
Введите текущий остаток квоты Google Drive в ГиБ. Включите `RUN_TAR_PLAN` для оценки или `RUN_TAR_BUILD` для упаковки. Используйте отдельный запуск для `test_unseen`. После отключения запустите упаковку с теми же split и каталогом.

In [ ]:
RUN_TAR_PLAN = False
RUN_TAR_BUILD = False

if RUN_TAR_PLAN or RUN_TAR_BUILD:
    if RUN_TAR_BUILD and drive_free_gib.value <= 0:
        raise ValueError('Введите текущий свободный объём из интерфейса Google Drive.')
    archive_report = build_tar_archives(
        DRIVE_PREPARED_ROOT, ARCHIVE_ROOT, TAR_STAGING_ROOT, selected_cache_splits(),
        archive_free_bytes=int(drive_free_gib.value * GIB) if drive_free_gib.value > 0 else None,
        plan_only=not RUN_TAR_BUILD,
    )
    display(pd.Series(archive_report))

## Подготовка локального кэша
Ячейка выполняет копирование синхронно и показывает прогресс. Повторный запуск пропускает проверенные FLAC. После ручного запуска верните переключатель в `False` перед общим запуском ячеек.

In [ ]:
RUN_CACHE_PREPARATION = False

if RUN_CACHE_PREPARATION:
    cache_splits = selected_cache_splits()
    CACHE_VERIFIED_SPLITS.difference_update(cache_splits)
    update_copy_progress(0, 1, {'copied': 0, 'skipped': 0, 'errors': 0})
    if cache_mode.value == 'tar':
        cache_result = restore_tar_cache(
            DRIVE_PREPARED_ROOT, ARCHIVE_ROOT, LOCAL_CACHE_ROOT, cache_splits,
            progress_callback=update_copy_progress,
        )
        display(pd.Series(cache_result))
    else:
        cache_result = prepare_local_cache(
            DRIVE_PREPARED_ROOT, LOCAL_CACHE_ROOT, cache_splits,
            verify_checksums=checksum_widget.value, progress_callback=update_copy_progress,
        )
        display(pd.Series(asdict(cache_result)))
        if not cache_result.ready:
            raise RuntimeError('Кэш не готов. Проверьте отчёт и повторите подготовку.')
    CACHE_VERIFIED_SPLITS.update(cache_splits)
    print('Кэш готов. Корень аудио', LOCAL_CACHE_ROOT)
else:
    print('Для копирования установите RUN_CACHE_PREPARATION = True и запустите эту ячейку.')

## Сравнение скорости переноса FLAC и TAR
Ячейка переносит во временный каталог самый маленький готовый train TAR и те же FLAC по одному. Замер может занять несколько минут.

In [ ]:
RUN_TAR_TRANSFER_BENCHMARK = False
if RUN_TAR_TRANSFER_BENCHMARK:
    display(pd.Series(benchmark_tar_transfer(DRIVE_PREPARED_ROOT, ARCHIVE_ROOT, TAR_STAGING_ROOT)))

## Настройки обучения CNN
Параметры совпадают с baseline notebook 02. Checkpoints всегда сохраняются в Google Drive, независимо от выбранного корня аудио.

In [ ]:
# 4. Интерактивная панель CNN и Mel
import json

from audio_deepfake_detector.src.cnn_training import (
    RuASDSegmentDataset,
    assert_checkpoint_compatible,
    compute_pos_weight,
    configs_from_mapping,
    create_spectrogram_figures,
    evaluate_checkpoint,
    make_loaders,
    resolve_run_directory,
    train_cnn,
)

controls = {
    'batch_size': widgets.Dropdown(options=[8, 16, 32, 64], value=32, description='Batch size'),
    'epochs': widgets.IntSlider(value=20, min=1, max=100, description='Epochs'),
    'learning_rate': widgets.Dropdown(options=[1e-4, 3e-4, 1e-3, 3e-3], value=1e-3, description='LR'),
    'weight_decay': widgets.FloatText(value=1e-4, description='Weight decay'),
    'early_stopping_patience': widgets.IntSlider(value=5, min=1, max=20, description='Patience'),
    'num_workers': widgets.Dropdown(options=[0, 1, 2, 4], value=2, description='Workers'),
    'seed': widgets.IntText(value=42, description='Seed'),
    'balance_classes': widgets.Checkbox(value=True, description='Балансировка pos_weight'),
    'use_amp': widgets.Checkbox(value=True, description='Mixed precision AMP'),
    'n_fft': widgets.Dropdown(options=[512, 1024, 2048], value=1024, description='FFT size'),
    'win_length': widgets.Dropdown(options=[512, 1024, 2048], value=1024, description='Window'),
    'hop_length': widgets.IntText(value=320, description='Hop length'),
    'n_mels': widgets.IntSlider(value=80, min=32, max=160, description='Mel bins'),
    'f_min': widgets.FloatText(value=20.0, description='Min Hz'),
    'f_max': widgets.FloatText(value=16000.0, description='Max Hz'),
}
run_id_widget = widgets.Text(value='baseline_cached_v1', description='run_id')
resume_widget = widgets.Checkbox(value=True, description='Resume training')
show_button = widgets.Button(description='Показать текущую конфигурацию', button_style='info')
config_output = widgets.Output()

def current_configs():
    return configs_from_mapping({name: widget.value for name, widget in controls.items()})

def show_configuration(_=None):
    with config_output:
        clear_output()
        try:
            mel_config, training_config = current_configs()
            run_dir = resolve_run_directory(TRAINING_ROOT, run_id_widget.value)
            data_root = active_data_root()
            weight = compute_pos_weight(segments)
            payload = {
                'audio_data_root': str(data_root),
                'checkpoint_run_directory': str(run_dir),
                'resume': resume_widget.value,
                'computed_pos_weight': weight,
                'mel_config': asdict(mel_config),
                'training_config': asdict(training_config),
            }
            print(json.dumps(payload, ensure_ascii=False, indent=2))
            checkpoint_path = run_dir / 'last_checkpoint.pth'
            if resume_widget.value and checkpoint_path.exists():
                checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
                assert_checkpoint_compatible(checkpoint, mel_config, training_config)
                print('Checkpoint совместим; продолжение с эпохи', checkpoint['epoch'] + 2)
        except Exception as error:  # noqa: BLE001 — сообщение выводится в панели
            print('ОШИБКА КОНФИГУРАЦИИ:', error)

show_button.on_click(show_configuration)
display(widgets.VBox([
    widgets.HTML('<b>Обучение</b>'),
    *[controls[key] for key in ['batch_size', 'epochs', 'learning_rate', 'weight_decay', 'early_stopping_patience', 'num_workers', 'seed', 'balance_classes', 'use_amp']],
    widgets.HTML('<b>Mel (sample rate: 32 000 Гц)</b>'),
    *[controls[key] for key in ['n_fft', 'win_length', 'hop_length', 'n_mels', 'f_min', 'f_max']],
    widgets.HTML('<b>Эксперимент</b>'), run_id_widget, resume_widget, show_button, config_output,
]))

## Диагностика узкого места
Benchmark не обучает модель. Он отдельно измеряет чтение байтов FLAC, декодирование, CPU log-Mel, получение batch, передачу на GPU и forward CNN. Сравнение Drive/кэша нужно выполнить в вашей Colab-сессии.

In [ ]:
# 5. Измерение Drive и локального кэша одинаковым тестом
from audio_deepfake_detector.src.training_benchmark import benchmark_roots

benchmark_button = widgets.Button(description='Сравнить Drive и локальный кэш', button_style='info')
benchmark_output = widgets.Output()

def run_benchmark(_):
    benchmark_button.disabled = True
    with benchmark_output:
        clear_output()
        try:
            mel_config, training_config = current_configs()
            local_root = require_ready_cache(
                DRIVE_PREPARED_ROOT, LOCAL_CACHE_ROOT, ('train', 'validation')
            )
            results = benchmark_roots(
                segments, {'Google Drive': DRIVE_PREPARED_ROOT, 'Local /content': local_root},
                mel_config, training_config, sample_count=16, batch_count=5,
            )
            display(results)
            print('Это измерение текущей сессии, а не гарантированное время полной эпохи.')
        except Exception as error:  # noqa: BLE001 — сообщение выводится в панели
            print('ОШИБКА BENCHMARK:', error)
        finally:
            benchmark_button.disabled = False

benchmark_button.on_click(run_benchmark)
display(benchmark_button, benchmark_output)

In [ ]:
# 6. Preflight одного train и validation batch
preflight_button = widgets.Button(description='Проверить один batch', button_style='warning')
preflight_output = widgets.Output()

def run_preflight(_):
    with preflight_output:
        clear_output()
        try:
            data_root = active_data_root()
            mel_config, training_config = current_configs()
            train_loader, validation_loader = make_loaders(segments, data_root, mel_config, training_config)
            for name, loader in [('train', train_loader), ('validation', validation_loader)]:
                log_mel, labels, sample_ids = next(iter(loader))
                print(name, 'root=', data_root, 'shape=', tuple(log_mel.shape),
                      'finite=', bool(torch.isfinite(log_mel).all()),
                      'labels=', sorted(labels.unique().tolist()), 'ids=', len(sample_ids))
            print('test_unseen не создавался и не читался.')
        except Exception as error:  # noqa: BLE001 — сообщение выводится в панели
            print('ОШИБКА PREFLIGHT:', error)

preflight_button.on_click(run_preflight)
display(preflight_button, preflight_output)

In [ ]:
# 7. Реальные log-Mel REAL/FAKE из выбранного корня
mel_button = widgets.Button(description='Показать реальные Mel', button_style='info')
mel_output = widgets.Output()

def show_real_mels(_):
    with mel_output:
        clear_output()
        try:
            data_root = active_data_root()
            mel_config, training_config = current_configs()
            dataset = RuASDSegmentDataset(segments, data_root, 'train', mel_config)
            for tag, figure in create_spectrogram_figures(dataset, training_config.spectrogram_examples_per_class):
                print(tag)
                display(figure)
        except Exception as error:  # noqa: BLE001 — сообщение выводится в панели
            print('ОШИБКА MEL:', error)

mel_button.on_click(show_real_mels)
display(mel_button, mel_output)

## TensorBoard, обучение и Resume
TensorBoard и checkpoints записываются на Drive. Для обучения установите `RUN_TRAINING = True` и запустите ячейку через ▶. Ячейка остаётся активной до завершения вызова `train_cnn()`. Ограничения времени работы Colab продолжают действовать.

In [ ]:
# 8. TensorBoard из постоянного каталога Drive
from tensorboard import notebook as tensorboard_notebook

RUNS_ROOT.mkdir(parents=True, exist_ok=True)
tensorboard_notebook.start(f'--logdir {RUNS_ROOT} --port 6006')

In [ ]:
# 9. Синхронный запуск обучения
RUN_TRAINING = False

if RUN_TRAINING:
    data_root = active_data_root(('train', 'validation'))
    mel_config, training_config = current_configs()
    run_dir = resolve_run_directory(TRAINING_ROOT, run_id_widget.value)
    print('Корень аудио', data_root)
    print('Checkpoints и TensorBoard', run_dir)
    _model, history = train_cnn(
        data_root, run_dir, mel_config, training_config, resume=resume_widget.value
    )
    print('Завершённых эпох', len(history))
else:
    print('Для обучения установите RUN_TRAINING = True и запустите эту ячейку.')

In [ ]:
# 10. История выбранного run_id
history_button = widgets.Button(description='Обновить историю')
history_output = widgets.Output()

def show_history(_):
    with history_output:
        clear_output()
        run_dir = resolve_run_directory(TRAINING_ROOT, run_id_widget.value)
        history_path = run_dir / 'training_history.json'
        if history_path.exists():
            payload = json.loads(history_path.read_text(encoding='utf-8'))
            history = payload.get('epochs', []) if isinstance(payload, dict) else payload
            display(pd.json_normalize(history))
        else:
            print('История ещё не создана:', history_path)

history_button.on_click(show_history)
display(history_button, history_output)

## Финальная оценка test_unseen
При включённом локальном кэше сначала отдельно отметьте `test_unseen` в панели кэша и подготовьте его. Эта кнопка не запускается автоматически.

In [ ]:
# 11. Отдельное действие финальной оценки
test_button = widgets.Button(description='ФИНАЛЬНО оценить test_unseen', button_style='danger')
test_output = widgets.Output()

def evaluate_test_unseen(_):
    with test_output:
        clear_output()
        try:
            data_root = active_data_root(('test_unseen',))
            _, training_config = current_configs()
            run_dir = resolve_run_directory(TRAINING_ROOT, run_id_widget.value)
            best_model = run_dir / 'best_model.pth'
            if not best_model.exists():
                raise FileNotFoundError(best_model)
            metrics = evaluate_checkpoint(
                data_root, best_model, split='test_unseen',
                batch_size=training_config.batch_size, num_workers=training_config.num_workers,
            )
            print(json.dumps(metrics, ensure_ascii=False, indent=2))
            (run_dir / 'test_unseen_metrics.json').write_text(
                json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8'
            )
        except Exception as error:  # noqa: BLE001 — сообщение выводится в панели
            print('ОШИБКА ФИНАЛЬНОЙ ОЦЕНКИ:', error)

test_button.on_click(evaluate_test_unseen)
display(test_button, test_output)